# SWYNEX Task 2: Data Cleaning and Basic Analysis
Clean and summarize a dataset using Python (pandas).

**To use your own dataset:** put your CSV in this folder and set `DATA_PATH` below. If no file is found, a sample messy sequence dataset is generated so the notebook still runs.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = 'dataset.csv'   # <- change to your dataset file name
SEQ_COL = 'sequence'        # <- name of the sequence column (if any)

## 1. Load data

In [ ]:
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print('Loaded', DATA_PATH)
else:
    rng = np.random.default_rng(42)
    n = 300
    def seq(L): return ''.join(rng.choice(list('ACGT'), L))
    df = pd.DataFrame({
        'sample_id': [f'S{i:04d}' for i in range(n)],
        'organism': rng.choice(['E. coli','e. coli ','Homo sapiens','homo sapiens','S. cerevisiae', None], n),
        'sequence': [seq(int(rng.integers(50, 400))) for _ in range(n)],
        'quality_score': rng.normal(32, 4, n).round(1),
        'read_count': rng.integers(100, 5000, n).astype(float),
    })
    df.loc[rng.choice(n, 15, replace=False), 'quality_score'] = np.nan
    df.loc[rng.choice(n, 10, replace=False), 'read_count'] = np.nan
    df.loc[rng.choice(n, 5, replace=False), 'quality_score'] = 99.9      # outliers
    df.loc[rng.choice(n, 6, replace=False), 'sequence'] = 'acgtNNXZ'     # invalid / lowercase
    df = pd.concat([df, df.sample(12, random_state=1)], ignore_index=True)  # duplicates
    print('Generated sample dataset')
df.head()

## 2. Inspect raw data

In [ ]:
print('Shape:', df.shape)
df.info()
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 3. Clean data

In [ ]:
raw_rows = len(df)
clean = df.copy()

# a) remove exact duplicates
clean = clean.drop_duplicates()

# b) standardize text columns (trim spaces, consistent case)
for c in clean.select_dtypes(exclude='number').columns:
    clean[c] = clean[c].str.strip()
if 'organism' in clean:
    clean['organism'] = clean['organism'].str.title()
if SEQ_COL in clean:
    clean[SEQ_COL] = clean[SEQ_COL].str.upper()
    # c) drop sequences with invalid characters (valid DNA: A, C, G, T)
    clean = clean[clean[SEQ_COL].str.fullmatch('[ACGT]+', na=False)]

# d) fill missing values: numeric -> median, categorical -> 'Unknown'
for c in clean.columns:
    if clean[c].dtype.kind in 'fi':
        clean[c] = clean[c].fillna(clean[c].median())
    else:
        clean[c] = clean[c].fillna('Unknown')

# e) remove outliers in quality_score using the IQR rule
if 'quality_score' in clean:
    q1, q3 = clean['quality_score'].quantile([.25, .75])
    iqr = q3 - q1
    clean = clean[clean['quality_score'].between(q1-1.5*iqr, q3+1.5*iqr)]

clean = clean.reset_index(drop=True)
print(f'Rows: {raw_rows} -> {len(clean)}  (removed {raw_rows-len(clean)})')

## 4. Sequence summaries

In [ ]:
if SEQ_COL in clean:
    clean['length'] = clean[SEQ_COL].str.len()
    clean['gc_content_%'] = clean[SEQ_COL].apply(lambda s: 100*(s.count('G')+s.count('C'))/len(s)).round(2)
clean.head()

## 5. Basic statistics

In [ ]:
summary = clean.describe().T
summary['median'] = clean.select_dtypes('number').median()
display(summary)
if 'organism' in clean:
    print(clean['organism'].value_counts())

## 6. Visualizations

In [ ]:
num = [c for c in ['length','gc_content_%','quality_score'] if c in clean]
fig, ax = plt.subplots(1, len(num), figsize=(5*len(num), 4))
for a, c in zip(np.atleast_1d(ax), num):
    a.hist(clean[c], bins=20, color='steelblue', edgecolor='white'); a.set_title(c)
plt.tight_layout(); plt.savefig('histograms.png', dpi=120); plt.show()

if 'organism' in clean:
    clean['organism'].value_counts().plot.bar(title='Samples per organism', color='seagreen')
    plt.tight_layout(); plt.savefig('organism_counts.png', dpi=120); plt.show()

## 7. Save results

In [ ]:
clean.to_csv('cleaned_dataset.csv', index=False)
summary.to_csv('summary_statistics.csv')
print('Saved cleaned_dataset.csv and summary_statistics.csv')

## Key learnings
- Removed duplicates, standardized text, and dropped invalid sequences.
- Filled missing numeric values with the median and categorical ones with 'Unknown'.
- Removed quality-score outliers with the IQR rule.
- Summarized length, GC content, and quality with statistics and plots.